# Copy the GitHub archive → Google Drive

Copies everything saved in `bhootbd/fao2` (`archive/ig_meme_templatesofficial`) into **My Drive / ig_meme_templatesofficial (Telegram archive)**, then renames the files in Drive to their descriptive names from `catalog.json`.

Click **Runtime → Run all** and allow Drive access. If it stops, run all again: files already in Drive are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/fao2
!git clone --depth 1 --filter=blob:none --sparse -b claude/pensive-newton-1m0wgi https://github.com/bhootbd/fao2 /content/fao2
!cd /content/fao2 && git sparse-checkout set archive/ig_meme_templatesofficial
!du -sh /content/fao2/archive/ig_meme_templatesofficial

In [ ]:
import json, os
DEST = '/content/drive/MyDrive/ig_meme_templatesofficial (Telegram archive)'
SRC = '/content/fao2/archive/ig_meme_templatesofficial'
media = DEST + '/media'
os.makedirs(media, exist_ok=True)
# Rename files already in Drive to their catalog names first, so the copy below doesn't duplicate them
catalog = json.load(open(SRC + '/catalog.json', encoding='utf-8'))
by_key = {}
for f in os.listdir(media):
    by_key.setdefault(f.split('__')[0].rsplit('.', 1)[0], []).append(f)
renamed = 0
for e in catalog:
    target = os.path.basename(e['file'])
    for f in by_key.get(e['key'], []):
        if f != target and not os.path.exists(os.path.join(media, target)):
            os.rename(os.path.join(media, f), os.path.join(media, target))
            renamed += 1
print('renamed in Drive:', renamed)

In [ ]:
# Delete duplicates that were removed from the archive (see removed.json)
removed_path = SRC + '/removed.json'
deleted = 0
if os.path.exists(removed_path):
    for r in json.load(open(removed_path, encoding='utf-8')):
        names = {r['file']} | {f for f in os.listdir(media) if r.get('key') and f.split('__')[0].rsplit('.', 1)[0] == r['key']}
        keep = r.get('duplicate_of')
        for f in names:
            if f != keep and os.path.exists(os.path.join(media, f)):
                os.remove(os.path.join(media, f)); deleted += 1
print('duplicates deleted from Drive:', deleted)

In [ ]:
!cp -rn "$SRC"/. "$DEST"/
!cp -f "$SRC"/index.json "$SRC"/catalog.json "$DEST"/
print('files in Drive:', len(os.listdir(media)), 'of', len(os.listdir(SRC + '/media')))
drive.flush_and_unmount()
print('all changes synced to Drive')